![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Optimize the Products Table</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Use Predictive Optimization, automatic liquid clustering, and time travel on the <code>products</code> table - with AI doing most of the work.</div>
</div>

</div>


## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.


In [0]:
%run ./Includes/Classroom-Setup-05

**Context.** In Lesson 04 you saw that on modern Databricks, table tuning is mostly **automatic**: Predictive Optimization runs compaction, statistics, and vacuum for you, and **automatic liquid clustering** chooses how to lay out the data. In this lab you'll set that up on the `products` table, then use **time travel** to move between versions after you change the table.

Product analytics typically filters by `category` first (turntables vs. amplifiers vs. speakers), then drills into `brand`. With automatic liquid clustering, you don't have to encode that yourself - Predictive Optimization learns it from how the table is queried.

In this practice you will:

1. Inspect `products` and confirm Predictive Optimization is on.
2. Turn on automatic liquid clustering with `CLUSTER BY AUTO`.
3. See the manual `OPTIMIZE` that PO runs for you (once, for understanding).
4. Change the table (add rows, add a column) and read its version history.
5. Use **time travel** to query an earlier version.

**How to work each task:** the default path is **Genie Code** - open it inline with **Cmd/Ctrl + I** in the target cell, or use the full **Genie / Assistant** panel, and give it the prompt provided. Run what it generates, then expand the reference to compare. You can always write the SQL yourself instead.


### Task 1: Inspect the table and confirm Predictive Optimization

First, look at the table's columns and storage details.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_703_1').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_703_1">Show the extended description of the products table, including its columns and storage details.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the full describe command</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb5').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb5">DESCRIBE EXTENDED products;</code></pre>
</div>
</details>

You should see the columns (`product_id`, `product_name`, `category`, `subcategory`, `brand`, `list_price`, `cost`, `release_year`) and a **Detailed Table Information** section with the storage `Location` and `Provider = delta`.

Now confirm Predictive Optimization is enabled, the same way you did in the demo:

1. In the left sidebar, click **Catalog**.
2. Navigate to your catalog &rarr; `vintage_audio` schema &rarr; `products` table.
3. On the **Overview**, find the **Predictive Optimization** field. If it shows enabled (often inherited from the catalog or schema), PO is already maintaining this table - compaction, statistics, and vacuum all happen automatically.


### Task 2: Turn on automatic liquid clustering

Enable automatic liquid clustering on `products`. With `CLUSTER BY AUTO`, Predictive Optimization picks and adapts the clustering keys from how the table is queried - you don't list columns yourself.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_703_2').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_703_2" style="white-space:pre-wrap; overflow-wrap:break-word;">Alter the products table to enable automatic liquid clustering so Databricks chooses the clustering keys automatically.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the full ALTER statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb6').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb6">ALTER TABLE products CLUSTER BY AUTO;</code></pre>
</div>
</details>

That's the whole setup. From here, PO clusters `products` in the background based on real query patterns (it will tend to favor `category` and `brand`, since those are what product queries filter on). No partitioning, no `ZORDER BY`, no key list to maintain.

> **If you already know the keys,** you can pin them with `CLUSTER BY (category, brand)` instead of `AUTO`. Automatic is the better default when the workload may change over time.


### Task 3: See the compaction PO runs for you

Predictive Optimization runs `OPTIMIZE` (file compaction) automatically. Run it once by hand just to see what that operation looks like and what it reports.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_703_3').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_703_3">Run OPTIMIZE on the products table to compact its files.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the full OPTIMIZE command</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb7').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb7">OPTIMIZE products;</code></pre>
</div>
</details>

The result reports compaction metrics. On this 250-row table they'll be tiny; on a production catalog with millions of products, this is meaningful work - which is exactly why you let PO handle it on a schedule instead of running it yourself. Statistics are likewise collected automatically as data is written, so there's no separate `ANALYZE` step to remember.


### Task 4: Change the table, then read its version history

A version history is only interesting once the table has actually changed. You'll make two changes - **add a couple of rows**, then **add a new column** and watch them show up as new versions.

First, run this cell to note the current version number (you'll come back to it for time travel):


In [0]:
%sql
-- The latest version number is the largest value in the 'version' column
DESCRIBE HISTORY products;

Note the highest **version** number shown - call it your *starting version*. Now add two clearly-labeled demo products.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_703_4').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_703_4" style="white-space:pre-wrap; overflow-wrap:break-word;">Insert two rows into the products table with product_id values DEMO-A and DEMO-B, product_name &#x27;Demo Turntable&#x27; and &#x27;Demo Amplifier&#x27;, category &#x27;Turntables&#x27; and &#x27;Amplifiers&#x27;, and a list_price of 999.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the full INSERT statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb8').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb8">INSERT INTO products (product_id, product_name, category, list_price) VALUES
  ('DEMO-A', 'Demo Turntable', 'Turntables', 999),
  ('DEMO-B', 'Demo Amplifier', 'Amplifiers', 999);</code></pre>
</div>
</details>

Next, add a new column. This is a **schema change**, and Delta tracks it as its own version. Add a `price_tier` column.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_703_5').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_703_5">Add a new string column called price_tier to the products table.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the full ALTER statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb9').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb9">ALTER TABLE products ADD COLUMNS (price_tier STRING);</code></pre>
</div>
</details>

In [0]:
%sql
-- Re-read the history to see your new versions
DESCRIBE HISTORY products;

Now the history is meaningful. Compared with your *starting version*, you should see new rows for the **`WRITE`** (the insert) and the **`ADD COLUMNS`** (the schema change).

> **Why this matters:** every version is a point you can audit or return to. That's what makes the next step - time travel possible.


### Task 5: Time travel to an earlier version

Delta lets you query the table **as it existed at an earlier version**, using `VERSION AS OF`. In Task 4's history, the version **before** you inserted the demo rows is **version 3**. Compare it against the current table.

Run the two cells below and compare the counts. The current table includes your two demo rows; version 3 does not.


In [0]:
%sql
-- Current row count (includes DEMO-A and DEMO-B)
SELECT COUNT(*) AS row_count_now FROM products;

Now query version 3, the version just before your inserts.


In [0]:
%sql
-- Row count as of version 3 (before the inserts)
SELECT COUNT(*) AS row_count_before FROM products VERSION AS OF 3;

`row_count_before` should be **2 lower** than `row_count_now` - the two demo rows you added aren't in version 3. You just queried the past without restoring anything. (If you wanted to actually roll the table back, you'd use `RESTORE TABLE products TO VERSION AS OF 3`.)

## Nice work

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">

In this lab you:

- Confirmed <strong>Predictive Optimization</strong> is maintaining the <code>products</code> table for you.
- Enabled <strong>automatic liquid clustering</strong> with <code>CLUSTER BY AUTO</code> - letting AI choose the data layout.
- Saw the <code>OPTIMIZE</code> compaction that PO runs automatically.
- Changed the table (added rows and a column) and read the resulting <strong>version history</strong>.
- Used <strong>time travel</strong> (<code>VERSION AS OF</code>) to query an earlier version.

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>